# 第九章 · 在T4上做强化学习（GRPO）

[![在Colab中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zxh0/vibe-writing/blob/main/toyllm_book2/code/book/ch09/colab.ipynb)

和前几章的笔记本一样，只做杂事：挂载Google Drive、拉代码，然后调用`book/ch09/`里的脚本。这一章不用下载数据：加法题都是现出的。

**起点是第七章训好的SFT模型**，要在Drive的`我的云端硬盘/toyllm2/runs/sft/model/`下。

分两步：

1. **冷启动**：用1到3位数的加法题做一次很短的SFT，教会模型“a + b = c”这个格式。每隔250步存一个模型目录，都在`toyllm2/runs/cold/`下。
2. **GRPO**：从冷启动的模型出发，让模型自己做2、3位数的题，只告诉它对还是错。每50步量一次1到5位数的正确率。

**用法**：菜单“代码执行程序 → 更改运行时类型”选T4 GPU，改好第一个格子里的配置，然后“全部运行”。
断线以后重新连上，再“全部运行”一次，训完的会跳过，`grpo.py`带着`--resume`，会从上一个checkpoint接着训。

In [ ]:
# ---- 配置 ----------------------------------------------------------------
REPO_URL = "https://github.com/zxh0/vibe-writing.git"
BRANCH = "main"
SUBDIR = "toyllm_book2/code"
DRIVE_DIR = "/content/drive/MyDrive/toyllm2"

BASE_RUN = "sft"          # 冷启动的起点：runs/<BASE_RUN>/model
COLD_RUN = "cold"         # 冷启动的记录：runs/<COLD_RUN>/
COLD_ARGS = []            # 额外的参数，比如 ["--steps", "2000"]
COLD_PICK = "model"       # GRPO从哪个冷启动模型出发：model（最后一步），或者step-250、step-500、step-750
RUN_NAME = "grpo"
TRAIN_ARGS = []           # 额外的参数，比如 ["--lr", "5e-6", "--digits", "3"]
RUN_EXPERIMENTS = True    # 对照实验：组的大小、有没有KL

In [ ]:
import os, pathlib, shutil, subprocess, sys

def sh(cmd, cwd=None):
    """跑一条shell命令，边跑边打印，失败就停。"""
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

import torch
assert torch.cuda.is_available(), "没有CUDA：菜单“代码执行程序 → 更改运行时类型”选GPU"
p = torch.cuda.get_device_properties(0)
cap = torch.cuda.get_device_capability()
print(f"GPU    {p.name}，sm{cap[0]}{cap[1]}，显存{p.total_memory / 2**30:.1f}GiB")
print(f"torch  {torch.__version__}，CUDA {torch.version.cuda}，Python {sys.version.split()[0]}")
print(f"CPU    {os.cpu_count()}个线程")
import safetensors, tokenizers   # 都是Colab自带的，缺了这里会报错

## 挂载Drive，拉代码

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = pathlib.Path(DRIVE_DIR)
RUNS = DRIVE / "runs"
BASE = RUNS / BASE_RUN / "model"
assert (BASE / "model.safetensors").exists(), f"Drive上没有{BASE}，先跑第七章"

# 稀疏克隆：整个仓库有好几本书，这里只取第九章的代码。
WORK = pathlib.Path("/content/vibe-writing")
if not WORK.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} --filter=blob:none --no-checkout {REPO_URL} {WORK}")
    sh(f"git sparse-checkout set --no-cone /{SUBDIR}/book/ch09/", cwd=WORK)
    sh("git checkout", cwd=WORK)
else:
    sh("git pull --ff-only", cwd=WORK)
sh("git log -1 --format='%h %ad %s' --date=short", cwd=WORK)
CODE = WORK / SUBDIR

def run(*args):
    """在code/下跑book/ch09/里的一个脚本，输出边跑边打印。"""
    cmd = [sys.executable, "-u", f"book/ch09/{args[0]}", *map(str, args[1:])]
    print("$ python", " ".join(cmd[2:]), flush=True)
    subprocess.run(cmd, cwd=CODE, check=True)

## 核对

KV缓存和不用缓存的结果一样；GRPO的优势、KL和一步更新的方向。都在CPU上算，几十秒。

In [ ]:
for f in ["model.safetensors", "config.json", "generation_config.json", "tokenizer.json"]:
    dst = CODE / "runs" / "sft" / "model" / f
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        shutil.copy(BASE / f, dst)
run("check_cache.py")
run("check_grpo.py")

## 冷启动

几分钟。训完的跳过；想重训，先删掉Drive上的`toyllm2/runs/cold/`。

In [ ]:
COLD = RUNS / COLD_RUN
if (COLD / "model").exists():
    print(f"{COLD_RUN}已经训完，跳过")
else:
    run("cold_start.py", "--name", COLD_RUN, "--init", BASE, "--runs", RUNS, *COLD_ARGS)
INIT = COLD / COLD_PICK
print("GRPO的起点：", INIT)

## 先看看起点

冷启动之后，每种位数贪婪解码做对多少，采样8次至少对一次的又有多少。后者是GRPO能拿到信号的地方。

In [ ]:
run("compare.py", INIT, "--n", 200)

## GRPO

断了重连以后，再跑一遍这个格子就会接着训。训完存成模型目录：`toyllm2/runs/<名字>/model/`。

In [ ]:
if (RUNS / RUN_NAME / "model").exists():
    print(f"{RUN_NAME}已经训完，跳过")
else:
    run("grpo.py", "--name", RUN_NAME, "--init", INIT, "--runs", RUNS,
        "--resume", "--export", *TRAIN_ARGS)
run("compare.py", INIT, RUNS / RUN_NAME / "model")

## 对照实验

各训200步，其余和完整训练一样：

* 组的大小：每步的回答总数不变（256个），一组4个回答、64道题，对比一组16个回答、16道题；
* 不要KL：`--beta 0`。

学习率按200步重新排，所以要和同样200步的默认设置比，`exp-base`就是它。

In [ ]:
SHORT = ["--steps", 200, "--eval-every", 50, "--ckpt-every", 50, "--export"]
EXPERIMENTS = {
    "exp-base": [],
    "exp-g4": ["--group", 4, "--prompts", 64],
    "exp-g16": ["--group", 16, "--prompts", 16],
    "exp-nokl": ["--beta", 0],
}
if RUN_EXPERIMENTS:
    for name, extra in EXPERIMENTS.items():
        if (RUNS / name / "model").exists():
            print(f"{name}已经训完，跳过")
            continue
        run("grpo.py", "--name", name, "--init", INIT, "--runs", RUNS,
            "--resume", *SHORT, *extra)